In [1]:
from pathlib import Path
raw_folder = Path(
    r"C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection"
    r"\data\raw\paderborn"
)
print("folder exists:", raw_folder.exists())

folder exists: True


In [2]:
#find the mat files
files  =sorted(raw_folder.glob("**/*.mat"))
print(len(files), "files found")

80 files found


In [3]:
#inspect the first 5 files
for file in files[:5]:
    print(file.name)


N09_M07_F10_K001_1.mat
N09_M07_F10_K001_10.mat
N09_M07_F10_K001_11.mat
N09_M07_F10_K001_12.mat
N09_M07_F10_K001_13.mat


In [4]:
#split the files names from its parts
recording_name = files[0].stem
parts = recording_name.split("_")
parts

['N09', 'M07', 'F10', 'K001', '1']

In [5]:
#extract bearning id and recording no
bearing_id  = parts[3]
recording_no = parts[4]

print("bearing id:", bearing_id)
print("recording no:", recording_no)

bearing id: K001
recording no: 1


In [6]:
#extract the operating condition code 
speed_code = parts[0]
torq_code = parts[1]
force_code = parts[2]
print("speed code:", speed_code)
print("torq code:", torq_code)
print("force code:", force_code)

speed code: N09
torq code: M07
force code: F10


In [7]:
record = {
    "bearing_id": bearing_id,
    "recording_no": recording_no,
    "speed_code": speed_code,
    "torq_code": torq_code,
    "force_code": force_code,
    "relative_path": files[0].relative_to(raw_folder).as_posix()
}

In [8]:
record

{'bearing_id': 'K001',
 'recording_no': '1',
 'speed_code': 'N09',
 'torq_code': 'M07',
 'force_code': 'F10',
 'relative_path': 'K001/K001/N09_M07_F10_K001_1.mat'}

In [9]:
#create registry of all recordings
records = []
for file in files:
    recording_name = file.stem
    parts = recording_name.split("_")
    bearing_id  = parts[3]
    recording_no = parts[4]
    speed_code = parts[0]
    torq_code = parts[1]
    force_code = parts[2]
    
    record = {
        "bearing_id": bearing_id,
        "recording_no": recording_no,
        "speed_code": speed_code,
        "torq_code": torq_code,
        "force_code": force_code,
        "relative_path": file.relative_to(raw_folder).as_posix()
    }
    
    records.append(record)
print("registry entries:", len(records))

registry entries: 80


In [12]:
#convert registry to a table
import pandas as pd
registry = pd.DataFrame(records)
print("rows:", registry.shape[0], "columns:", registry.shape[1])
registry.head()

rows: 80 columns: 6


,bearing_id,recording_no,speed_code,torq_code,force_code,relative_path
0,K001,1,N09,M07,F10,K001/K001/N09_M07_F10_K001_1.mat
1,K001,10,N09,M07,F10,K001/K001/N09_M07_F10_K001_10.mat
2,K001,11,N09,M07,F10,K001/K001/N09_M07_F10_K001_11.mat
3,K001,12,N09,M07,F10,K001/K001/N09_M07_F10_K001_12.mat
4,K001,13,N09,M07,F10,K001/K001/N09_M07_F10_K001_13.mat


In [13]:
registry['recording_id']=[Path(path).stem for path in registry["relative_path"]]

In [14]:
registry.head()

,bearing_id,recording_no,speed_code,torq_code,force_code,relative_path,recording_id
0,K001,1,N09,M07,F10,K001/K001/N09_M07_F10_K001_1.mat,N09_M07_F10_K001_1
1,K001,10,N09,M07,F10,K001/K001/N09_M07_F10_K001_10.mat,N09_M07_F10_K001_10
2,K001,11,N09,M07,F10,K001/K001/N09_M07_F10_K001_11.mat,N09_M07_F10_K001_11
3,K001,12,N09,M07,F10,K001/K001/N09_M07_F10_K001_12.mat,N09_M07_F10_K001_12
4,K001,13,N09,M07,F10,K001/K001/N09_M07_F10_K001_13.mat,N09_M07_F10_K001_13


In [15]:
#check to duplicate recording ids
duplicate_recording_ids = registry['recording_id'].duplicated()
print("duplicate recording ids:", duplicate_recording_ids.sum())


duplicate recording ids: 0


In [16]:
#count recordings of each bearing
registry['bearing_id'].value_counts()

bearing_id
K001    80
Name: count, dtype: int64

In [17]:
#counts recording of each operating condition
condition_counts = registry.groupby(['speed_code', 'torq_code', 'force_code']).size()
print(condition_counts)

speed_code  torq_code  force_code
N09         M07        F10           20
N15         M01        F10           20
            M07        F04           20
                       F10           20
dtype: int64


In [18]:
registry = registry.sort_values(by=['bearing_id', 'speed_code', 'torq_code', 'force_code', 'recording_no']).reset_index(drop=True)
registry.head()

,bearing_id,recording_no,speed_code,torq_code,force_code,relative_path,recording_id
0,K001,1,N09,M07,F10,K001/K001/N09_M07_F10_K001_1.mat,N09_M07_F10_K001_1
1,K001,10,N09,M07,F10,K001/K001/N09_M07_F10_K001_10.mat,N09_M07_F10_K001_10
2,K001,11,N09,M07,F10,K001/K001/N09_M07_F10_K001_11.mat,N09_M07_F10_K001_11
3,K001,12,N09,M07,F10,K001/K001/N09_M07_F10_K001_12.mat,N09_M07_F10_K001_12
4,K001,13,N09,M07,F10,K001/K001/N09_M07_F10_K001_13.mat,N09_M07_F10_K001_13


In [19]:
registry["recording_no"] = registry["recording_no"].astype(int)

registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

print(registry[["recording_no", "recording_id"]].head())

   recording_no        recording_id
0             1  N09_M07_F10_K001_1
1             2  N09_M07_F10_K001_2
2             3  N09_M07_F10_K001_3
3             4  N09_M07_F10_K001_4
4             5  N09_M07_F10_K001_5


In [20]:
registry["condition"] = registry["bearing_id"].map({"K001": "healthy"})
registry["partition"] = registry["bearing_id"].map({"K001": "development"})

print(registry[["bearing_id", "condition", "partition"]].drop_duplicates())

  bearing_id condition    partition
0       K001   healthy  development


In [21]:
output_path = Path(
    r"C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection"
    r"\metadata\recording_registry.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
registry.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection\metadata\recording_registry.csv


In [22]:
saved_registry = pd.read_csv(output_path)

print("Rows and columns:", saved_registry.shape)
print(saved_registry.groupby(["bearing_id", "condition", "partition"]).size())

Rows and columns: (80, 9)
bearing_id  condition  partition  
K001        healthy    development    80
dtype: int64


In [23]:
k002_files = list(raw_folder.rglob("*_K002_*.mat"))

print("K002 recordings found:", len(k002_files))

K002 recordings found: 80


In [24]:
files = sorted(raw_folder.rglob("*.mat"))

print("Total recordings:", len(files))

Total recordings: 160


In [25]:
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print("Rows and columns:", registry.shape)
print(registry["bearing_id"].value_counts())

Rows and columns: (160, 7)
bearing_id
K001    80
K002    80
Name: count, dtype: int64


In [26]:
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development"
})

print(registry.groupby(["bearing_id", "condition", "partition"]).size())

bearing_id  condition  partition  
K001        healthy    development    80
K002        healthy    development    80
dtype: int64


In [27]:
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())
print("Missing values:", registry.isna().sum().sum())

Duplicate IDs: 0
Missing values: 0


In [28]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

registry.to_csv(output_path, index=False)

print("Saved rows:", len(registry))
print("Location:", output_path)

Saved rows: 160
Location: C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection\metadata\recording_registry.csv


In [30]:
ka04_files = list(raw_folder.rglob("*_KA04_*.mat"))

print("KA04 recordings found:", len(ka04_files))


KA04 recordings found: 80


In [32]:
files = sorted(raw_folder.rglob("*.mat"))

print("Total recordings:", len(files))

Total recordings: 240


In [33]:
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print(registry["bearing_id"].value_counts())

bearing_id
K001    80
K002    80
KA04    80
Name: count, dtype: int64


In [34]:
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "KA04": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development"
})

print(registry.groupby(["bearing_id", "condition", "partition"]).size())

bearing_id  condition  partition  
K001        healthy    development    80
K002        healthy    development    80
KA04        faulty     development    80
dtype: int64


In [35]:
print("Rows and columns:", registry.shape)
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())
print("Missing values:", registry.isna().sum().sum())

Rows and columns: (240, 9)
Duplicate IDs: 0
Missing values: 0


In [36]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

registry.to_csv(output_path, index=False)

print("Saved rows:", len(registry))

Saved rows: 240


In [37]:
ki04_files = list(raw_folder.rglob("*_KI04_*.mat"))

print("KI04 recordings found:", len(ki04_files))

KI04 recordings found: 80


In [38]:
files = sorted(raw_folder.rglob("*.mat"))
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print("Total recordings:", len(registry))
print(registry["bearing_id"].value_counts())

Total recordings: 320
bearing_id
K001    80
K002    80
KA04    80
KI04    80
Name: count, dtype: int64


In [39]:
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "KA04": "faulty",
    "KI04": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development"
})

print(registry.groupby(["bearing_id", "condition", "partition"]).size())

bearing_id  condition  partition  
K001        healthy    development    80
K002        healthy    development    80
KA04        faulty     development    80
KI04        faulty     development    80
dtype: int64


In [40]:
print("Rows and columns:", registry.shape)
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())
print("Missing values:", registry.isna().sum().sum())

Rows and columns: (320, 9)
Duplicate IDs: 0
Missing values: 0


In [41]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

registry.to_csv(output_path, index=False)

print("Saved rows:", len(registry))

Saved rows: 320


In [42]:
k003_files = list(raw_folder.rglob("*_K003_*.mat"))

print("K003 recordings found:", len(k003_files))

K003 recordings found: 80


In [43]:
ka15_files = list(raw_folder.rglob("*_KA15_*.mat"))

print("KA15 recordings found:", len(ka15_files))

KA15 recordings found: 80


In [44]:
ki14_files = list(raw_folder.rglob("*_KI14_*.mat"))

print("KI14 recordings found:", len(ki14_files))

KI14 recordings found: 80


In [45]:
files = sorted(raw_folder.rglob("*.mat"))
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print("Total recordings:", len(registry))
print(registry["bearing_id"].value_counts().sort_index())

Total recordings: 560
bearing_id
K001    80
K002    80
K003    80
KA04    80
KA15    80
KI04    80
KI14    80
Name: count, dtype: int64


In [46]:
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

partition    condition
development  faulty       160
             healthy      160
final_test   faulty       160
             healthy       80
dtype: int64


In [47]:
print("Missing values:", registry.isna().sum().sum())
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())

partitions_per_bearing = registry.groupby("bearing_id")["partition"].nunique()
print("Partitions per bearing:")
print(partitions_per_bearing)

Missing values: 0
Duplicate IDs: 0
Partitions per bearing:
bearing_id
K001    1
K002    1
K003    1
KA04    1
KA15    1
KI04    1
KI14    1
Name: partition, dtype: int64


In [48]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

registry.to_csv(output_path, index=False)

print("Saved rows:", len(registry))

Saved rows: 560
